In [9]:
import re

print("=" * 60)
print("SEARCH QUERY SPELLING CORRECTOR")
print("=" * 60)


# ------------------------------------------------------------
# 1. LOAD BIRKBECK CORPUS
# ------------------------------------------------------------

corpus_file = "birkbeck.dat"

correct_words = set()
misspelling_map = {}

with open(corpus_file, "r", encoding="utf-8", errors="ignore") as file:

    current_correct_word = None

    for line in file:

        line = line.strip()

        if not line:
            continue

        # Correct word
        if line.startswith("$"):

            current_correct_word = line[1:].lower()
            correct_words.add(current_correct_word)

        # Misspelled word
        else:

            if current_correct_word is not None:

                misspelled_word = line.lower()

                misspelling_map[misspelled_word] = current_correct_word


print("\nBirkbeck Corpus Loaded Successfully")
print("Correct Words:", len(correct_words))
print("Misspellings:", len(misspelling_map))


# ------------------------------------------------------------
# 2. CHECK BIRKBECK MAPPINGS
# ------------------------------------------------------------

print("\nBirkbeck Mapping Check:")

print("machne ->", misspelling_map.get("machne", "Not found"))
print("lerning ->", misspelling_map.get("lerning", "Not found"))
print("cours ->", misspelling_map.get("cours", "Not found"))


# ------------------------------------------------------------
# 3. EDIT DISTANCE
# ------------------------------------------------------------

def edit_distance(word1, word2):

    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = [[0] * cols for _ in range(rows)]

    # First column
    for i in range(rows):
        matrix[i][0] = i

    # First row
    for j in range(cols):
        matrix[0][j] = j

    # Fill matrix
    for i in range(1, rows):

        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,        # Deletion
                matrix[i][j - 1] + 1,        # Insertion
                matrix[i - 1][j - 1] + cost  # Substitution
            )

    return matrix[-1][-1]


# ------------------------------------------------------------
# 4. FIND CORRECT WORD
# ------------------------------------------------------------

def correct_word(word):

    word = word.lower()

    # --------------------------------------------------------
    # Case 1: Already a correct word
    # --------------------------------------------------------

    if word in correct_words:
        return word


    # --------------------------------------------------------
    # Case 2: Known Birkbeck misspelling
    # --------------------------------------------------------

    if word in misspelling_map:
        return misspelling_map[word]


    # --------------------------------------------------------
    # Case 3: Edit-distance fallback
    # --------------------------------------------------------

    candidates = []

    for candidate in correct_words:

        # Ignore candidates with very different lengths
        if abs(len(word) - len(candidate)) > 2:
            continue

        distance = edit_distance(word, candidate)

        candidates.append(
            (
                distance,
                abs(len(word) - len(candidate)),
                candidate
            )
        )


    # No suitable candidate
    if not candidates:
        return word


    # --------------------------------------------------------
    # Sort by:
    # 1. Smallest edit distance
    # 2. Smallest length difference
    # 3. Alphabetical order
    # --------------------------------------------------------

    candidates.sort(
        key=lambda x: (x[0], x[1], x[2])
    )

    return candidates[0][2]


# ------------------------------------------------------------
# 5. CORRECT SEARCH QUERY
# ------------------------------------------------------------

def correct_query(query):

    words = re.findall(r"[a-zA-Z_]+", query.lower())

    corrected_words = []
    corrections = []

    for word in words:

        corrected_word = correct_word(word)

        corrected_words.append(corrected_word)

        if word != corrected_word:

            corrections.append(
                (word, corrected_word)
            )

    corrected_query = " ".join(corrected_words)

    return corrected_query, corrections


# ------------------------------------------------------------
# 6. USER INPUT
# ------------------------------------------------------------

print("\n" + "-" * 60)

query = input("Enter your search query: ")


# ------------------------------------------------------------
# 7. PROCESS QUERY
# ------------------------------------------------------------

corrected_query, corrections = correct_query(query)


# ------------------------------------------------------------
# 8. DISPLAY RESULT
# ------------------------------------------------------------

print("\nOriginal Query:")
print(query)

print("\nCorrections:")

if corrections:

    for wrong, correct in corrections:
        print(wrong, "->", correct)

else:

    print("No spelling errors found")


print("\nCorrected Query:")
print(corrected_query)


# ------------------------------------------------------------
# 9. COMPLETION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SPELLING CORRECTION COMPLETED")
print("=" * 60)

SEARCH QUERY SPELLING CORRECTOR

Birkbeck Corpus Loaded Successfully
Correct Words: 6130
Misspellings: 33968

Birkbeck Mapping Check:
machne -> Not found
lerning -> learning
cours -> courses

------------------------------------------------------------


Enter your search query:  machne lerning cours



Original Query:
machne lerning cours

Corrections:
machne -> mache
lerning -> learning
cours -> courses

Corrected Query:
mache learning courses

SPELLING CORRECTION COMPLETED
